# Training YOLO26n — dataset v3 expanded

Questo notebook usa lo split leakage-safe del v3 canonico e aggiunge al solo Train le 18 immagini PhenoCam revisionate presenti in `dataset-v3-expanded`. Validation, TEST-ID e TEST-OOD restano identici al benchmark canonico.

Non allena sulle 240 immagini operative e non usa il checkpoint v2, che potrebbe aver visto immagini ora assegnate a TEST-ID. Il modello conserva tutte le 80 classi COCO richieste dal runtime.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil
import sys

import torch
import yaml
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "phenocam").is_dir():
    raise RuntimeError("Avvia il notebook dalla radice del repository o da notebooks/")
sys.path.insert(0, str(ROOT))

from dataset.builder.partition import verify_artifact

CANONICAL = ROOT / "dataset" / "dataset-v3"
EXPANDED = ROOT / "dataset" / "dataset-v3-expanded"
BASE_MODEL = ROOT / "models" / "yolo26n.pt"
WORK = ROOT / "output" / "training-v3-expanded"
RUNS = WORK / "runs"
V3_PT = ROOT / "models" / "yolo26n-v3-expanded.pt"
V3_ONNX = ROOT / "models" / "yolo26n-v3-expanded.onnx"
EPOCHS, IMAGE_SIZE, BATCH, SEED = 30, 640, 8, 42
DEVICE = "mps" if torch.backends.mps.is_available() else 0 if torch.cuda.is_available() else "cpu"

if not BASE_MODEL.is_file():
    raise RuntimeError("Checkpoint COCO models/yolo26n.pt assente")
WORK.mkdir(parents=True, exist_ok=True)
print({"torch": torch.__version__, "device": DEVICE, "epochs": EPOCHS, "batch": BATCH})

## Verifica dei due artefatti

Il v3 canonico passa il verificatore completo. L'artefatto expanded viene controllato contro il proprio manifest SHA-256 prima di confrontare le identità.

In [ ]:
verification = verify_artifact(CANONICAL)
if verification["status"] != "passed" or verification["split_images"] != {"train": 1600, "val": 200, "test_id": 200, "test_ood": 240}:
    raise RuntimeError("Il dataset v3 non rispetta il contratto canonico")

checksum_manifest = EXPANDED / "metadata" / "checksums.sha256"
expanded_root = EXPANDED.resolve()
for line in checksum_manifest.read_text(encoding="utf-8").splitlines():
    expected, relative = line.split("  ", 1)
    candidate = (EXPANDED / relative).resolve()
    if Path(relative).is_absolute() or not candidate.is_relative_to(expanded_root) or not candidate.is_file():
        raise RuntimeError("Percorso non valido nel manifest expanded")
    digest = hashlib.sha256()
    with candidate.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise RuntimeError("Checksum expanded non valido")
print({"canonical": verification["status"], "expanded_checksums": "passed"})

## Composizione Train expanded

Le nuove identità devono essere esattamente 18, pubbliche, revisionate e appartenenti a siti PhenoCam già assegnati al Train canonico. Il notebook scrive soltanto una lista di immagini sotto `output/`; non modifica i dataset.

In [ ]:
def read_manifest(root):
    with (root / "metadata/source-images.csv").open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

canonical_rows = read_manifest(CANONICAL)
expanded_rows = read_manifest(EXPANDED)
canonical_ids = {row["image_id"] for row in canonical_rows}
expanded_ids = {row["image_id"] for row in expanded_rows}
if len(canonical_ids) != len(canonical_rows) or len(expanded_ids) != len(expanded_rows) or not canonical_ids.issubset(expanded_ids):
    raise RuntimeError("Identità duplicate o base canonica incompleta")
additions = [row for row in expanded_rows if row["image_id"] not in canonical_ids]
if len(additions) != 18 or any(row["source_dataset"] != "phenocam" or row["cohort"] != "public_teacher_reviewed" or row["split"] != "train" or not row["label_path"] for row in additions):
    raise RuntimeError("L'espansione pubblica non rispetta il contratto da 18 immagini")

site_splits = {}
for row in canonical_rows:
    if row["site_id"]:
        site_splits.setdefault(row["site_id"], set()).add(row["split"])
addition_sites = {row["site_id"] for row in additions}
if any(site_splits.get(site) != {"train"} for site in addition_sites):
    raise RuntimeError("Un sito expanded non appartiene interamente al Train canonico")

canonical_train = [row for row in canonical_rows if row["split"] == "train"]
training_images = [(CANONICAL / row["image_path"]).resolve() for row in canonical_train]
training_images.extend((EXPANDED / row["image_path"]).resolve() for row in additions)
if len(training_images) != 1618 or len(set(training_images)) != 1618 or any(not path.is_file() for path in training_images):
    raise RuntimeError("Train expanded incompleto o duplicato")
train_list = WORK / "train.txt"
train_list.write_text("\n".join(map(str, training_images)) + "\n", encoding="utf-8")
print({"canonical_train": len(canonical_train), "additions": len(additions), "sites": sorted(addition_sites), "training_images": len(training_images)})

## Configurazione e training

Validation e test puntano sempre al v3 canonico. Il miglior checkpoint su Validation viene copiato in `models/yolo26n-v3-expanded.pt`.

In [ ]:
checkpoint = YOLO(BASE_MODEL)
coco_names = dict(checkpoint.names)
target_names = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
if len(coco_names) != 80 or any(coco_names[class_id] != name for class_id, name in target_names.items()):
    raise RuntimeError("Il checkpoint non espone l'inventario COCO atteso")
data_yaml = WORK / "dataset-v3-expanded-80-classes.yaml"
data_yaml.write_text(yaml.safe_dump({
    "train": str(train_list),
    "val": str(CANONICAL / "images/val"),
    "test": [str(CANONICAL / "images/test/id"), str(CANONICAL / "images/test/ood")],
    "names": coco_names,
}, sort_keys=False), encoding="utf-8")

model = YOLO(BASE_MODEL)
model.train(
    data=str(data_yaml), epochs=EPOCHS, patience=8, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, cache=False, seed=SEED, deterministic=True,
    project=str(RUNS), name="yolo26n-v3-expanded", exist_ok=True,
)
best_checkpoint = Path(model.trainer.best).resolve()
if not best_checkpoint.is_file():
    raise RuntimeError("Il training non ha prodotto best.pt")
shutil.copy2(best_checkpoint, V3_PT)
trained = YOLO(V3_PT)
if dict(trained.names) != coco_names:
    raise RuntimeError("Il modello expanded non conserva le 80 classi COCO")
print(f"Checkpoint expanded: {V3_PT} ({V3_PT.stat().st_size / 1024 / 1024:.1f} MiB)")

## Metriche finali

I risultati sono direttamente confrontabili con v3 perché Validation, TEST-ID e TEST-OOD sono identici. I test non devono guidare ulteriori decisioni di training.

In [ ]:
def summarize(metrics):
    return {
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "per_class_mAP50-95": {target_names[class_id]: float(metrics.box.maps[class_id]) for class_id in target_names},
    }

evaluation = {}
for split_name, split_path in (("validation", CANONICAL / "images/val"), ("test-id", CANONICAL / "images/test/id"), ("test-ood", CANONICAL / "images/test/ood")):
    split_yaml = WORK / f"{split_name}.yaml"
    split_yaml.write_text(yaml.safe_dump({"val": str(split_path), "names": coco_names}, sort_keys=False), encoding="utf-8")
    metrics = trained.val(data=str(split_yaml), split="val", imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE, workers=0, plots=True, project=str(RUNS), name=split_name, exist_ok=True)
    evaluation[split_name] = summarize(metrics)
(WORK / "metrics-v3-expanded.json").write_text(json.dumps(evaluation, indent=2) + "\n", encoding="utf-8")
print(json.dumps(evaluation, indent=2))

## Export ONNX e contratto runtime

In [ ]:
exported = Path(trained.export(format="onnx", opset=20, imgsz=IMAGE_SIZE, batch=1, dynamic=False)).resolve()
if exported != V3_ONNX.resolve():
    shutil.copy2(exported, V3_ONNX)

from phenocam.classes.selection import enabled_class_names, model_class_ids
from phenocam.inference.pipeline import process_image
from phenocam.inference.runtime import create_session, model_contract

session = create_session(V3_ONNX)
_, _, width, height, onnx_names = model_contract(session)
model_class_ids(onnx_names, enabled_class_names())
if onnx_names != coco_names or (width, height) != (IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError("Contratto ONNX expanded inatteso")
sample_input = next((CANONICAL / "images/test/ood").glob("*.jpg"))
sample_output = WORK / "runtime-sample.jpg"
elapsed = process_image(V3_ONNX, sample_input, sample_output, None)
print({"onnx": str(V3_ONNX), "sample": str(sample_output), "seconds": round(elapsed, 3)})

## Curve di training

In [ ]:
import matplotlib.pyplot as plt

history_path = RUNS / "yolo26n-v3-expanded" / "results.csv"
with history_path.open(newline="", encoding="utf-8") as stream:
    history = list(csv.DictReader(stream))
epochs = [int(row["epoch"]) for row in history]
map_all = [float(row["metrics/mAP50-95(B)"]) for row in history]
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [float(row["metrics/mAP50(B)"]) for row in history], label="mAP50")
axes[0].plot(epochs, map_all, label="mAP50-95")
axes[0].set(title="Qualità su Validation", xlabel="Epoch", ylabel="mAP", ylim=(0, 1)); axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].plot(epochs, [float(row["train/box_loss"]) for row in history], label="box train")
axes[1].plot(epochs, [float(row["val/box_loss"]) for row in history], label="box validation", linestyle="--")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Loss"); axes[1].legend(); axes[1].grid(alpha=0.2)
figure.tight_layout()
figure.savefig(WORK / "training-curves.png", dpi=150, bbox_inches="tight")
plt.show()